In [1]:
%load_ext dotenv
%dotenv

In [2]:
from qdrant_client import QdrantClient
#from qdrant_client.models import VectorParams, Distance
from llama_index.core import VectorStoreIndex, Settings, Document, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from datasets import load_dataset
import os
import logging

logging.basicConfig(level=logging.INFO)

In [3]:
client = QdrantClient(path=os.getenv('QDRANT_PATH'))

In [ ]:
"""
client = QdrantClient(
    url=f"{os.getenv('QDRANT_HOST')}:{os.getenv('QDRANT_PORT')}",
    api_key=os.getenv('QDRANT_API_KEY'),
    )
"""

In [ ]:
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    device="mps",
    embed_batch_size=10,
)

Settings.embed_model = embed_model
Settings.chunk_size = 512
Settings.chunk_overlap = 50

In [6]:
def rebuild_index(client: QdrantClient):
    try:
        news_dataset = load_dataset(
            "RealTimeData/bbc_news_alltime", "2017-12", split="train"
        )
        logging.info(f"Loaded the BBC News dataset with {len(news_dataset)} rows")
        logging.info(f"Successfully loaded the BBC News dataset with {len(news_dataset)} rows.")
    except Exception as e:
        raise ValueError(f"Error loading the BBC News dataset: {str(e)}")

    news_articles = news_dataset["content"]
    unique_articles = set()
    for article in news_articles:
        if article:
            unique_articles.add(article)
    unique_news_articles = list(unique_articles)
    logging.info(f"We have {len(unique_news_articles)} unique articles in our database.")

    articles = [article for article in unique_news_articles if article and len(article) <= 50000]

    documents = [Document(text=t) for t in articles]
    vector_store = QdrantVectorStore(client=client, collection_name=os.getenv('QDRANT_COLLECTION'))
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(documents, storage_context=storage_context, show_progress=True)
    return index

In [7]:
if client.collection_exists(os.getenv('QDRANT_COLLECTION')):
    vector_store = QdrantVectorStore(client=client, collection_name=os.getenv('QDRANT_COLLECTION'))
    index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)
else:
    index = rebuild_index(client)

retriever = index.as_retriever(
    similarity_top_k=5,
)

In [ ]:
print(os.getenv('QDRANT_COLLECTION'))
client.collection_exists(collection_name=os.getenv('QDRANT_COLLECTION'))

In [ ]:
response = retriever.retrieve("Trading tariff between China and US")
for r in response:
    print("\n----------\n")
    print(r)